# M3 · ②~⑤ 업종 × 규모 기술통계와 관계 분석

M2 `dashboard_metrics.parquet`의 300개 집계 셀을 분석한다. **목적은 분포와 비교 대상 확인**이며, 사업장당 사고확률·인과효과를 추정하지 않는다. 그래프 HTML과 분석 요약을 개인 폴더에 저장한다.

In [ ]:
from pathlib import Path
import json, re, unicodedata
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists()), None)
assert ROOT is not None, 'SANUP-P 프로젝트에서 실행하세요.'
PERSONAL = ROOT/'data'/'personal'
(PERSONAL/'processed').mkdir(parents=True, exist_ok=True)
(PERSONAL/'reports').mkdir(parents=True, exist_ok=True)

from scipy.stats import spearmanr
import plotly.express as px
data_path=PERSONAL/'processed'/'dashboard_metrics.parquet'
assert data_path.exists(), 'M2 대시보드 전처리 노트북을 먼저 실행하세요.'
df=pd.read_parquet(data_path)
assert len(df)==300 and df['reference_date'].nunique()==1
FIGURES=PERSONAL/'figures'; FIGURES.mkdir(parents=True,exist_ok=True)

## 1. 전체 분포와 결측 확인

In [ ]:
metrics=['accident_injured','accident_deaths','establishments','fatality_rate']
summary=df[metrics].describe(percentiles=[.25,.5,.75]).T
summary['missing']=df[metrics].isna().sum()
display(summary)
print('대업종',df.industry_major.nunique(),'중분류',df.industry_middle.nunique(),
      '규모',df.size_code.nunique(),'결측',df[metrics].isna().sum().to_dict())

## 2. 규모별 분포 · 비율의 평균은 전국 사망만인율이 아님

In [ ]:
size_order=['lt5','5_9','10_19','20_29','30_49','50_99','100_299','300_499','500_999','1000_plus']
by_size=df.groupby('size_code',observed=True).agg(
    cells=('industry_middle','size'),
    injured_median=('accident_injured','median'),
    deaths_median=('accident_deaths','median'),
    establishments_median=('establishments','median'),
    fatality_rate_median=('fatality_rate','median'),
    rate_missing=('fatality_rate',lambda x: int(x.isna().sum()))
).reindex(size_order)
display(by_size)
fig=px.box(df,x='size_code',y='fatality_rate',category_orders={'size_code':size_order},
           points='outliers',title='2025 업종 중분류별 사망만인율 분포 · 규모별')
fig.update_layout(xaxis_title='사업장 규모',yaxis_title='공식 사망만인율')
fig.write_html(FIGURES/'dashboard_fatality_rate_by_size.html',include_plotlyjs='directory')

## 3. 같은 업종·규모 셀에서 값의 관계를 탐색

In [ ]:
correlations=[]
for a,b in [('accident_injured','accident_deaths'),('establishments','accident_injured')]:
    pair=df[[a,b]].dropna()
    rho,_=spearmanr(pair[a],pair[b])
    correlations.append({'metric_a':a,'metric_b':b,'n_cells':len(pair),'spearman_rho':float(rho)})
display(pd.DataFrame(correlations))
display(df.nlargest(8,'fatality_rate')[['industry_major','industry_middle','size_code',
                                       'fatality_rate','accident_deaths','establishments']])
fig=px.scatter(df,x='establishments',y='accident_injured',color='size_code',
               hover_data=['industry_major','industry_middle'],
               title='2025 사업장 수와 사고재해자수 · 업종×규모 집계 셀')
fig.write_html(FIGURES/'dashboard_establishments_vs_injured.html',include_plotlyjs='directory')

## 4. 수치와 해석을 함께 보고서로 저장

In [ ]:
lines=['# ②~⑤ 대시보드 기술통계','',
       f'- 기준일: {df.reference_date.iloc[0]}',
       f'- 업종×규모 집계 셀: {len(df)}개',
       f'- 사망만인율 결측: {int(df.fatality_rate.isna().sum())}개',
       '', '## 탐색적 순위상관']
for row in correlations:
    lines.append(f"- {row['metric_a']} ↔ {row['metric_b']}: Spearman ρ={row['spearman_rho']:.3f}, n={row['n_cells']}")
lines += ['', '## 해석 한계',
          '- 각 행은 사고가 아닌 업종×규모 집계 셀이다. 상관은 인과효과가 아니다.',
          '- 규모별 사망만인율 중앙값은 업종별 수치의 비가중 중앙값이며 전국 공식률이 아니다.',
          '- 사업장 수를 근로자 수로 취급하거나 사망만인율의 분모로 사용하지 않는다.',
          '- 일부 광업 셀의 공식 사망만인율이 매우 크고 사고사망자수와 바로 대응하지 않는다. 원래 지표 정의를 확인하기 전에는 원인으로 설명하지 않는다.',
          '- ① SIF 사례와 ②~⑤ 2025 집계치를 동일 기간 자료처럼 직접 결합하지 않는다.']
report=PERSONAL/'reports'/'dashboard_eda.md'
report.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('보고서:',report)

## ②~⑤ 비교 그래프

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Image, display

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
if any(font.name=='Malgun Gothic' for font in font_manager.fontManager.ttflist):
    plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
figures=ROOT/'data'/'personal'/'figures'; figures.mkdir(parents=True,exist_ok=True)
chart_df=pd.read_parquet(ROOT/'data'/'personal'/'processed'/'dashboard_metrics.parquet')
assert len(chart_df)==300

for column,label,color,filename in [
    ('accident_injured','사고재해자수 (명)','#1f6e8c','dashboard_injured_by_industry.png'),
    ('accident_deaths','사고사망자수 (명)','#b65f38','dashboard_deaths_by_industry.png'),
]:
    values=chart_df.groupby('industry_major')[column].sum().sort_values()
    fig,ax=plt.subplots(figsize=(10,6))
    ax.barh(values.index.astype(str),values.values,color=color)
    ax.set(xlabel=label,title=f'2025 업종 대분류별 {label}')
    ax.bar_label(ax.containers[0],fmt='{:,.0f}',padding=3)
    fig.tight_layout()
    path=figures/filename
    fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)
    display(Image(filename=str(path)))

size_order=['lt5','5_9','10_19','20_29','30_49','50_99','100_299','300_499','500_999','1000_plus']
rate_data=[np.log1p(chart_df.loc[chart_df.size_code==size,'fatality_rate'].dropna().to_numpy()) for size in size_order]
fig,ax=plt.subplots(figsize=(11,5))
ax.boxplot(rate_data,tick_labels=size_order,showfliers=True)
ax.set(xlabel='사업장 규모',ylabel='log(1 + 공식 사망만인율)',
       title='2025 업종 중분류별 사망만인율 분포 · 결측 14칸 제외')
ax.tick_params(axis='x',rotation=35)
fig.tight_layout()
path=figures/'dashboard_fatality_rate_distribution.png'
fig.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)
display(Image(filename=str(path)))

In [ ]:
qa={
    '사망만인율 결측':int(df.fatality_rate.isna().sum()),
    '사고사망자 0명·사망만인율 양수':int(((df.accident_deaths==0)&(df.fatality_rate>0)).sum()),
    '사망만인율 1000 초과':int((df.fatality_rate>1000).sum()),
    '사업장 0개소·사고재해자 양수':int(((df.establishments==0)&(df.accident_injured>0)).sum()),
}
display(pd.DataFrame([qa]))
display(df.loc[(df.establishments==0)&(df.accident_injured>0),
    ['industry_major','industry_middle','size_code','accident_injured','establishments']])
display(df.assign(rate_missing=df.fatality_rate.isna()).groupby('size_code',observed=True)
        .rate_missing.sum().rename('사망만인율 결측 칸').to_frame())

## 통계 결과와 해석

- ②~⑤를 긴 형식으로 바꿔 업종 중분류×규모 300칸을 만들었다. ⑤ 사망만인율은 14칸이 비어 있어 286칸 기준으로 요약했다. 공란은 원자료 설명상 산출 불가인 경우라 0으로 채우지 않았다.
- 사고재해자수와 사고사망자수의 Spearman 순위상관은 0.734, 사업장수와 사고재해자수는 0.783이었다. 서로 다른 업종·규모가 섞인 집계자료의 관계다. 한 변수가 다른 변수를 일으킨다는 뜻은 아니다.
- ③ 사고사망자가 0인데 ⑤ 사망만인율은 양수인 칸이 70개, ⑤가 1,000을 넘는 칸이 8개(광업), ④ 사업장수가 0인데 ② 재해자가 1명인 칸이 1개 있었다. 원본과 대조해 발견사항으로 남겼고 임의로 맞추거나 지우지 않았다.
- 특히 사망만인율은 임금근로자 1만 명당 사망자 지표이고 사업장 수는 그 분모가 아니다. 사망만인율의 높은 값은 규모가 작은 집단에서 분모가 작을 때 크게 움직일 수 있으므로 원자료 정의와 해당 칸을 함께 확인한다.
- 300칸은 모든 업종에서 같은 노출량·관측 조건을 보장하지 않는다. 평균·중앙값을 전국 공식률처럼 표현하지 말고, 대시보드에서는 업종과 규모의 원래 칸을 보존한다.

### 다음 분석·발표에서 사용할 내용

- 상관은 탐색적 참고값으로만 쓰고 인과 해석은 하지 않는다.
- 극단값·공란·분모 정의를 그래프 설명에 표시한다. ③ 사고사망자 수와 ⑤ 사망만인율을 같은 값처럼 섞어 합산하지 않는다.
- M2의 `dashboard_metrics.parquet`을 읽어 재현할 수 있다. 공식 표와 다르면 로컬 전처리 요약 및 원본 CSV를 다시 대조한다.
